# Hierarchical Product & SKU Demand Forecasting

## Project Objective
Build a two-level demand forecasting system that combines Product-Level and SKU-Level forecasting to improve production planning.

## Forecasting Architecture

**Model 1 — Product-Level Forecasting**
- Forecast monthly demand for all products.
- Identify the Top 20 products based on total profit contribution.
- Use Product-Level predictions as the total demand anchor.

**Model 2 — SKU-Level Forecasting**
- Forecast monthly demand independently for each SKU.
- Focus on SKUs belonging to the selected Top 20 products.
- Calculate each SKU's share of its parent product's predicted demand.

**Forecast Reconciliation**
- Calculate SKU-level demand proportions.
- Distribute Product-Level forecasts across their corresponding SKUs.
- Ensure that the sum of allocated SKU forecasts equals the Product-Level forecast.

## Model Evaluation
Compare:
1. Direct Product-Level Forecasting.
2. Direct SKU-Level Forecasting.
3. Product Forecast Distributed Across SKUs.

Evaluation metrics:
- WAPE
- MAE
- RMSE
- Bias

## Business Outcome
Generate monthly and quarterly production requirements at both Product and SKU levels.

In [26]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Machine Learning
from catboost import CatBoostRegressor

# Model Evaluation
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# Model Persistence
import joblib

# File Management
from pathlib import Path

# Date and Time
from datetime import datetime

# Warnings
import warnings
warnings.filterwarnings("ignore")

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Visualization Settings
sns.set_theme(style="whitegrid")

print("All libraries imported successfully.")

All libraries imported successfully.


In [27]:
DATA_DIR = Path("data")
MODEL_DIR = Path("models")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

sales_items = pd.read_csv(DATA_DIR / "sales_order_items.csv")
products = pd.read_csv(DATA_DIR / "products_catalog.csv")
branches = pd.read_csv(DATA_DIR / "branches.csv")

print("Datasets loaded successfully.")

print(f"Sales Items: {sales_items.shape}")
print(f"Products: {products.shape}")
print(f"Branches: {branches.shape}")

Datasets loaded successfully.
Sales Items: (1138536, 13)
Products: (139, 6)
Branches: (80, 7)


In [28]:
# Convert order_date to datetime
sales_items["order_date"] = pd.to_datetime(sales_items["order_date"])

# Ensure IDs have consistent data types
sales_items["product_id"] = sales_items["product_id"].astype(str)
sales_items["sku_id"] = sales_items["sku_id"].astype(str)
products["product_id"] = products["product_id"].astype(str)

# Create monthly period
sales_items["month"] = (
    sales_items["order_date"].dt.to_period("M").dt.to_timestamp()
)

# Identify the last completed month
latest_data_date = sales_items["order_date"].max()

last_completed_month = (
    latest_data_date.to_period("M").to_timestamp()
    - pd.offsets.MonthBegin(1)
)

print(f"Latest data date: {latest_data_date}")
print(f"Last completed month: {last_completed_month}")

Latest data date: 2026-10-18 00:00:00
Last completed month: 2026-09-01 00:00:00


In [29]:
# Calculate total margin for each product
product_margin = (
    sales_items
    .groupby("product_id", as_index=False)["margin_EGP"]
    .sum()
    .sort_values("margin_EGP", ascending=False)
)

# Select the top 20 products
top_20_products = product_margin.head(20).copy()

# Get product IDs
top_20_product_ids = top_20_products["product_id"].tolist()

# Add product names
top_20_products = top_20_products.merge(
    products[["product_id", "product_name_ar", "category_en"]],
    on="product_id",
    how="left"
)

display(top_20_products)

,product_id,margin_EGP,product_name_ar,category_en
0,PR0060,"24,029,662.06",3 قطع بوكسر رجالى مارسريزيه,Men boxers
1,PR0090,"14,662,698.68",8 قطع بوكسر CN,Men boxers
2,PR0121,"11,752,566.67",تيشيرت مرسريزيه,Men tops
3,PR0081,"10,791,195.16",4 اندرشيرت كم استريتش,Men tops
4,PR0085,"10,350,852.39",5 قطع بوكسر كاجوال,Men boxers
5,PR0122,"9,674,418.33",بيجامة شتوي حريمي كود 2031,Women pajamas
6,PR0123,"9,569,254.22",بيجامة شتوي حريمي كود 2032,Women pajamas
7,PR0137,"8,963,952.17",تيشيرت مرسريزيه الوان ( رمادي - اسود - ذهبي- ا...,Men tops
8,PR0086,"8,615,756.71",6 قطع بوكسر ديجيتال,Men boxers
9,PR0120,"7,935,567.78",بوكسر مرسريزيه,Men boxers


In [30]:
# Filter sales data for the top 20 products
top_20_sku_data = sales_items[
    sales_items["product_id"].isin(top_20_product_ids)
].copy()

# Get unique SKU-Product combinations
top_20_skus = (
    top_20_sku_data[["product_id", "sku_id"]]
    .drop_duplicates()
    .sort_values(["product_id", "sku_id"])
    .reset_index(drop=True)
)

# Summary
print(f"Number of selected products: {top_20_skus['product_id'].nunique()}")
print(f"Number of selected SKUs: {top_20_skus['sku_id'].nunique()}")

display(top_20_skus.head(10))

Number of selected products: 20
Number of selected SKUs: 100


,product_id,sku_id
0,PR0059,SKU000291
1,PR0059,SKU000292
2,PR0059,SKU000293
3,PR0059,SKU000294
4,PR0059,SKU000295
5,PR0060,SKU000296
6,PR0060,SKU000297
7,PR0060,SKU000298
8,PR0060,SKU000299
9,PR0060,SKU000300


In [31]:
# Aggregate monthly demand for all products
monthly_product_actual = (
    sales_items
    .groupby(["product_id", "month"], as_index=False)
    .agg(
        gross_quantity_demanded=("quantity", "sum")
    )
)

# Keep only completed months
monthly_product_actual = monthly_product_actual[
    monthly_product_actual["month"] <= last_completed_month
].copy()

print(f"Monthly product records: {len(monthly_product_actual):,}")
print(f"Unique products: {monthly_product_actual['product_id'].nunique():,}")
print(f"Date range: {monthly_product_actual['month'].min()} to "
      f"{monthly_product_actual['month'].max()}")

display(monthly_product_actual.head())

Monthly product records: 13,344
Unique products: 139
Date range: 2018-10-01 00:00:00 to 2026-09-01 00:00:00


,product_id,month,gross_quantity_demanded
0,PR0001,2018-10-01,12
1,PR0001,2018-11-01,36
2,PR0001,2018-12-01,77
3,PR0001,2019-01-01,37
4,PR0001,2019-02-01,20


In [32]:
monthly_sku_actual = (
    top_20_sku_data
    .groupby(["product_id", "sku_id", "month"], as_index=False)
    .agg(
        gross_quantity_demanded=("quantity", "sum")
    )
)

# Keep only completed months
monthly_sku_actual = monthly_sku_actual[
    monthly_sku_actual["month"] <= last_completed_month
].copy()

print(f"Monthly SKU records: {len(monthly_sku_actual):,}")
print(f"Unique SKUs: {monthly_sku_actual['sku_id'].nunique():,}")
print(f"Unique products: {monthly_sku_actual['product_id'].nunique():,}")

display(monthly_sku_actual.head())

Monthly SKU records: 9,600
Unique SKUs: 100
Unique products: 20


,product_id,sku_id,month,gross_quantity_demanded
0,PR0059,SKU000291,2018-10-01,11
1,PR0059,SKU000291,2018-11-01,53
2,PR0059,SKU000291,2018-12-01,110
3,PR0059,SKU000291,2019-01-01,55
4,PR0059,SKU000291,2019-02-01,36


In [33]:
# Define test period
test_start = pd.Timestamp("2026-01-01")
test_months = pd.date_range(
    start=test_start,
    end=last_completed_month,
    freq="MS"
)

# Product-level test actuals
product_test_actual = (
    pd.MultiIndex.from_product(
        [top_20_product_ids, test_months],
        names=["product_id", "month"]
    )
    .to_frame(index=False)
    .merge(
        monthly_product_actual,
        on=["product_id", "month"],
        how="left"
    )
)

product_test_actual["gross_quantity_demanded"] = (
    product_test_actual["gross_quantity_demanded"].fillna(0)
)

# SKU-level test actuals
sku_product_pairs = top_20_skus[["product_id", "sku_id"]].drop_duplicates()

sku_test_actual = (
    sku_product_pairs.assign(_key=1)
    .merge(
        pd.DataFrame({"month": test_months, "_key": 1}),
        on="_key"
    )
    .drop(columns="_key")
    .merge(
        monthly_sku_actual,
        on=["product_id", "sku_id", "month"],
        how="left"
    )
)

sku_test_actual["gross_quantity_demanded"] = (
    sku_test_actual["gross_quantity_demanded"].fillna(0)
)

print(f"Test months: {len(test_months)}")
print(f"Product test records: {len(product_test_actual):,}")
print(f"SKU test records: {len(sku_test_actual):,}")

display(product_test_actual.head())
display(sku_test_actual.head())

Test months: 9
Product test records: 180
SKU test records: 900


,product_id,month,gross_quantity_demanded
0,PR0060,2026-01-01,784
1,PR0060,2026-02-01,744
2,PR0060,2026-03-01,1032
3,PR0060,2026-04-01,924
4,PR0060,2026-05-01,871


,product_id,sku_id,month,gross_quantity_demanded
0,PR0059,SKU000291,2026-01-01,186
1,PR0059,SKU000291,2026-02-01,77
2,PR0059,SKU000291,2026-03-01,205
3,PR0059,SKU000291,2026-04-01,221
4,PR0059,SKU000291,2026-05-01,149


In [34]:
import joblib

# Load model artifacts
product_model_artifact = joblib.load(
    MODEL_DIR / "product_demand_model.joblib"
)

sku_model_artifact = joblib.load(
    MODEL_DIR / "sku_demand_model.joblib"
)

# Extract product model and metadata
product_model = product_model_artifact["model"]
product_feature_columns = product_model_artifact["feature_columns"]
product_categorical_features = product_model_artifact["categorical_features"]
product_numerical_features = product_model_artifact["numerical_features"]
product_target = product_model_artifact["target"]

# Extract SKU model and metadata
sku_model = sku_model_artifact["model"]
sku_feature_columns = sku_model_artifact["feature_columns"]
sku_categorical_features = sku_model_artifact["categorical_features"]
sku_numerical_features = sku_model_artifact["numerical_features"]
sku_target = sku_model_artifact["target"]

print("Both models loaded successfully.")

print(f"Product Model: {type(product_model).__name__}")
print(f"SKU Model: {type(sku_model).__name__}")

print(f"\nProduct features: {len(product_feature_columns)}")
print(f"SKU features: {len(sku_feature_columns)}")

Both models loaded successfully.
Product Model: CatBoostRegressor
SKU Model: CatBoostRegressor

Product features: 22
SKU features: 22


In [35]:
print("PRODUCT MODEL FEATURES")
print("=" * 50)

for i, feature in enumerate(product_feature_columns, 1):
    print(f"{i:02d}. {feature}")

print("\nSKU MODEL FEATURES")
print("=" * 50)

for i, feature in enumerate(sku_feature_columns, 1):
    print(f"{i:02d}. {feature}")

PRODUCT MODEL FEATURES
01. product_id
02. category_en
03. audience
04. listed_price_EGP
05. active_branches
06. year
07. month_number
08. quarter
09. month_sin
10. month_cos
11. lag_1
12. lag_2
13. lag_3
14. lag_6
15. lag_12
16. lag_13
17. rolling_mean_3
18. rolling_std_3
19. rolling_mean_6
20. rolling_std_6
21. rolling_mean_12
22. rolling_std_12

SKU MODEL FEATURES
01. product_id
02. sku_id
03. category_en
04. audience
05. active_branches
06. year
07. month_number
08. quarter
09. month_sin
10. month_cos
11. lag_1
12. lag_2
13. lag_3
14. lag_6
15. lag_12
16. lag_13
17. rolling_mean_3
18. rolling_std_3
19. rolling_mean_6
20. rolling_std_6
21. rolling_mean_12
22. rolling_std_12


In [36]:
print("PRODUCT ARTIFACT KEYS:")
print(list(product_model_artifact.keys()))

print("\nSKU ARTIFACT KEYS:")
print(list(sku_model_artifact.keys()))

print("\nPRODUCT MODEL FEATURE NAMES:")
if hasattr(product_model, "feature_names_"):
    print(product_model.feature_names_)
else:
    print("Feature names are not stored in the model.")

print("\nSKU MODEL FEATURE NAMES:")
if hasattr(sku_model, "feature_names_"):
    print(sku_model.feature_names_)
else:
    print("Feature names are not stored in the model.")

PRODUCT ARTIFACT KEYS:
['model', 'best_params', 'feature_columns', 'categorical_features', 'numerical_features', 'target', 'correction_factor', 'average_bias']

SKU ARTIFACT KEYS:
['model', 'best_params', 'feature_columns', 'categorical_features', 'numerical_features', 'target', 'correction_factor', 'average_bias']

PRODUCT MODEL FEATURE NAMES:
['product_id', 'category_en', 'audience', 'listed_price_EGP', 'active_branches', 'year', 'month_number', 'quarter', 'month_sin', 'month_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12']

SKU MODEL FEATURE NAMES:
['product_id', 'sku_id', 'category_en', 'audience', 'active_branches', 'year', 'month_number', 'quarter', 'month_sin', 'month_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12']


In [37]:
print("PRODUCT CATALOG COLUMNS:")
print(products.columns.tolist())

print("\nPRODUCT MODEL REQUIRED CATEGORICAL FEATURES:")
print(product_categorical_features)

print("\nPRODUCT MODEL REQUIRED NUMERICAL FEATURES:")
print(product_numerical_features)

PRODUCT CATALOG COLUMNS:
['product_id', 'product_name_ar', 'category_en', 'audience', 'listed_price_EGP', 'source_url']

PRODUCT MODEL REQUIRED CATEGORICAL FEATURES:
['product_id', 'category_en', 'audience']

PRODUCT MODEL REQUIRED NUMERICAL FEATURES:
['listed_price_EGP', 'active_branches', 'year', 'month_number', 'quarter', 'month_sin', 'month_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12']


In [38]:
print("SKU MODEL REQUIRED CATEGORICAL FEATURES:")
print(sku_categorical_features)

print("\nSKU MODEL REQUIRED NUMERICAL FEATURES:")
print(sku_numerical_features)

print("\nAVAILABLE SKU DATA COLUMNS:")
print(sales_items.columns.tolist())

print("\nAVAILABLE BRANCH DATA COLUMNS:")
print(branches.columns.tolist())

SKU MODEL REQUIRED CATEGORICAL FEATURES:
['product_id', 'sku_id', 'category_en', 'audience']

SKU MODEL REQUIRED NUMERICAL FEATURES:
['active_branches', 'year', 'month_number', 'quarter', 'month_sin', 'month_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12']

AVAILABLE SKU DATA COLUMNS:
['sales_line_id', 'sales_order_id', 'order_date', 'branch_id', 'sku_id', 'product_id', 'quantity', 'unit_price_EGP', 'gross_line_EGP', 'discount_line_EGP', 'net_line_EGP', 'unit_cost_EGP', 'margin_EGP', 'month']

AVAILABLE BRANCH DATA COLUMNS:
['branch_id', 'branch_name_ar', 'governorate_ar', 'city_ar', 'address_ar', 'retail_location_type', 'opening_date']


In [39]:
# Filter historical demand for the selected products
product_demand_history = monthly_product_actual[
    monthly_product_actual["product_id"].isin(top_20_product_ids)
].copy()

# Sort chronologically
product_demand_history = product_demand_history.sort_values(
    ["product_id", "month"]
).reset_index(drop=True)

print("Product demand history prepared.")
print(f"Records: {len(product_demand_history):,}")
print(f"Products: {product_demand_history['product_id'].nunique()}")

display(product_demand_history.head())

Product demand history prepared.
Records: 1,920
Products: 20


,product_id,month,gross_quantity_demanded
0,PR0059,2018-10-01,136
1,PR0059,2018-11-01,217
2,PR0059,2018-12-01,407
3,PR0059,2019-01-01,376
4,PR0059,2019-02-01,198


In [40]:
# Create product-month combinations for the test period
product_test_features = (
    pd.MultiIndex.from_product(
        [top_20_product_ids, test_months],
        names=["product_id", "month"]
    )
    .to_frame(index=False)
)

# Merge product metadata
product_test_features = product_test_features.merge(
    products[
        ["product_id", "category_en", "audience", "listed_price_EGP"]
    ].drop_duplicates("product_id"),
    on="product_id",
    how="left"
)

# Calendar features
product_test_features["year"] = (
    product_test_features["month"].dt.year
)

product_test_features["month_number"] = (
    product_test_features["month"].dt.month
)

product_test_features["quarter"] = (
    product_test_features["month"].dt.quarter
)

product_test_features["month_sin"] = np.sin(
    2 * np.pi * product_test_features["month_number"] / 12
)

product_test_features["month_cos"] = np.cos(
    2 * np.pi * product_test_features["month_number"] / 12
)

print("Base product features created.")
print(product_test_features.shape)

display(product_test_features.head())

Base product features created.
(180, 10)


,product_id,month,category_en,audience,listed_price_EGP,year,month_number,quarter,month_sin,month_cos
0,PR0060,2026-01-01,Men boxers,Unisex,2270,2026,1,1,0.50,0.87
1,PR0060,2026-02-01,Men boxers,Unisex,2270,2026,2,1,0.87,0.50
2,PR0060,2026-03-01,Men boxers,Unisex,2270,2026,3,1,1.00,0.00
3,PR0060,2026-04-01,Men boxers,Unisex,2270,2026,4,2,0.87,-0.50
4,PR0060,2026-05-01,Men boxers,Unisex,2270,2026,5,2,0.50,-0.87


In [41]:
# Convert opening_date to datetime
branches["opening_date"] = pd.to_datetime(
    branches["opening_date"]
)

# Calculate active branches for each test month
active_branches = pd.DataFrame({
    "month": test_months
})

active_branches["active_branches"] = active_branches["month"].apply(
    lambda month: (
        branches["opening_date"] <= month + pd.offsets.MonthEnd(0)
    ).sum()
)

# Merge into product features
product_test_features = product_test_features.merge(
    active_branches,
    on="month",
    how="left"
)

display(active_branches)

,month,active_branches
0,2026-01-01,80
1,2026-02-01,80
2,2026-03-01,80
3,2026-04-01,80
4,2026-05-01,80
5,2026-06-01,80
6,2026-07-01,80
7,2026-08-01,80
8,2026-09-01,80


In [42]:
# Check missing required features
missing_features = [
    feature
    for feature in product_feature_columns
    if feature not in product_test_features.columns
]

print("Available features:", len(product_test_features.columns))
print("Required features:", len(product_feature_columns))

print("\nMissing features:")
print(missing_features)

print("\nMissing values in available features:")
display(
    product_test_features.isna().sum()[
        product_test_features.isna().sum() > 0
    ]
)

Available features: 11
Required features: 22

Missing features:
['lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12']

Missing values in available features:


Series([], dtype: int64)

In [43]:
# Combine historical and test periods
product_feature_data = pd.concat([
    product_demand_history[
        ["product_id", "month", "gross_quantity_demanded"]
    ],
    product_test_features[["product_id", "month"]].assign(
        gross_quantity_demanded=np.nan
    )
], ignore_index=True)

product_feature_data = product_feature_data.sort_values(
    ["product_id", "month"]
).reset_index(drop=True)

# Calculate lag and rolling features
for lag in [1, 2, 3, 6, 12, 13]:
    product_feature_data[f"lag_{lag}"] = (
        product_feature_data
        .groupby("product_id")["gross_quantity_demanded"]
        .shift(lag)
    )

for window in [3, 6, 12]:
    shifted_demand = (
        product_feature_data
        .groupby("product_id")["gross_quantity_demanded"]
        .shift(1)
    )

    product_feature_data[f"rolling_mean_{window}"] = (
        shifted_demand
        .groupby(product_feature_data["product_id"])
        .transform(
            lambda x: x.rolling(window, min_periods=window).mean()
        )
    )

    product_feature_data[f"rolling_std_{window}"] = (
        shifted_demand
        .groupby(product_feature_data["product_id"])
        .transform(
            lambda x: x.rolling(window, min_periods=window).std()
        )
    )

# Keep only test-period features
product_lag_features = product_feature_data[
    product_feature_data["month"].isin(test_months)
].copy()

# Merge into test features
product_test_features = product_test_features.merge(
    product_lag_features.drop(
        columns=["gross_quantity_demanded"]
    ),
    on=["product_id", "month"],
    how="left"
)

print("Lag and rolling features generated.")
display(product_test_features.head())

Lag and rolling features generated.


,product_id,month,category_en,audience,listed_price_EGP,year,month_number,quarter,month_sin,month_cos,active_branches,lag_1,lag_2,lag_3,lag_6,lag_12,lag_13,rolling_mean_3,rolling_std_3,rolling_mean_6,rolling_std_6,rolling_mean_12,rolling_std_12
0,PR0060,2026-01-01,Men boxers,Unisex,2270,2026,1,1,0.50,0.87,80,959.00,894.00,970.00,837.00,781.00,787.00,941.00,41.07,867.50,88.23,814.75,108.68
1,PR0060,2026-01-01,Men boxers,Unisex,2270,2026,1,1,0.50,0.87,80,784.00,959.00,894.00,759.00,587.00,781.00,879.00,88.46,858.67,94.34,815.00,108.60
2,PR0060,2026-02-01,Men boxers,Unisex,2270,2026,2,1,0.87,0.50,80,NaN,784.00,959.00,786.00,890.00,587.00,NaN,NaN,NaN,NaN,NaN,NaN
3,PR0060,2026-02-01,Men boxers,Unisex,2270,2026,2,1,0.87,0.50,80,744.00,NaN,784.00,970.00,848.00,890.00,NaN,NaN,NaN,NaN,NaN,NaN
4,PR0060,2026-03-01,Men boxers,Unisex,2270,2026,3,1,1.00,0.00,80,NaN,744.00,NaN,894.00,757.00,848.00,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
# Check required feature availability
missing_features = [
    feature
    for feature in product_feature_columns
    if feature not in product_test_features.columns
]

# Check missing values
missing_values = (
    product_test_features[product_feature_columns]
    .isna()
    .sum()
)

print("Missing columns:", missing_features)

print("\nMissing values:")
display(missing_values[missing_values > 0])

print("\nFeature matrix shape:")
print(product_test_features[product_feature_columns].shape)

Missing columns: []

Missing values:


lag_1              160
lag_2              160
lag_3              140
lag_6              120
lag_12              60
lag_13              40
rolling_mean_3     320
rolling_std_3      320
rolling_mean_6     320
rolling_std_6      320
rolling_mean_12    320
rolling_std_12     320
dtype: int64


Feature matrix shape:
(360, 22)


In [45]:
print("PRODUCT FEATURES:")
print(product_feature_columns)

print("\nPRODUCT CATEGORICAL:")
print(product_categorical_features)

print("\nSKU FEATURES:")
print(sku_feature_columns)

print("\nSKU CATEGORICAL:")
print(sku_categorical_features)

print("\nPRODUCT CORRECTION:")
print("Factor:", product_model_artifact["correction_factor"])
print("Bias:", product_model_artifact["average_bias"])

print("\nSKU CORRECTION:")
print("Factor:", sku_model_artifact["correction_factor"])
print("Bias:", sku_model_artifact["average_bias"])

PRODUCT FEATURES:
['product_id', 'category_en', 'audience', 'listed_price_EGP', 'active_branches', 'year', 'month_number', 'quarter', 'month_sin', 'month_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12']

PRODUCT CATEGORICAL:
['product_id', 'category_en', 'audience']

SKU FEATURES:
['product_id', 'sku_id', 'category_en', 'audience', 'active_branches', 'year', 'month_number', 'quarter', 'month_sin', 'month_cos', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_13', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12']

SKU CATEGORICAL:
['product_id', 'sku_id', 'category_en', 'audience']

PRODUCT CORRECTION:
Factor: 1.0535509790508806
Bias: -5.355097905088053

SKU CORRECTION:
Factor: 1.09939349357912
Bias: -9.939349357912018


In [46]:

# ============================================================
# 1. PREPARE HISTORICAL MONTHLY PANELS
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

TARGET = "gross_quantity_demanded"

history_start = monthly_product_actual["month"].min()

all_months = pd.date_range(
    start=history_start,
    end=last_completed_month,
    freq="MS"
)

# ------------------------------------------------------------
# Product-level monthly panel
# ------------------------------------------------------------

product_grid = pd.MultiIndex.from_product(
    [top_20_product_ids, all_months],
    names=["product_id", "month"]
).to_frame(index=False)

product_panel = product_grid.merge(
    monthly_product_actual[
        ["product_id", "month", TARGET]
    ],
    on=["product_id", "month"],
    how="left"
)

product_panel[TARGET] = product_panel[TARGET].fillna(0)

product_panel = product_panel.sort_values(
    ["product_id", "month"]
).reset_index(drop=True)


# ------------------------------------------------------------
# SKU-level monthly panel
# ------------------------------------------------------------

sku_product_pairs = (
    top_20_skus[["product_id", "sku_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

sku_grid = (
    sku_product_pairs.assign(_key=1)
    .merge(
        pd.DataFrame({"month": all_months, "_key": 1}),
        on="_key"
    )
    .drop(columns="_key")
)

sku_panel = sku_grid.merge(
    monthly_sku_actual[
        ["product_id", "sku_id", "month", TARGET]
    ],
    on=["product_id", "sku_id", "month"],
    how="left"
)

sku_panel[TARGET] = sku_panel[TARGET].fillna(0)

sku_panel = sku_panel.sort_values(
    ["product_id", "sku_id", "month"]
).reset_index(drop=True)


# ============================================================
# 2. PREPARE CALENDAR AND ACTIVE BRANCH FEATURES
# ============================================================

branches["opening_date"] = pd.to_datetime(
    branches["opening_date"]
)

active_branch_table = pd.DataFrame({
    "month": all_months
})

active_branch_table["active_branches"] = (
    active_branch_table["month"].apply(
        lambda month: (
            branches["opening_date"]
            <= month + pd.offsets.MonthEnd(0)
        ).sum()
    )
)


def add_calendar_features(df):
    df = df.copy()

    df["year"] = df["month"].dt.year
    df["month_number"] = df["month"].dt.month
    df["quarter"] = df["month"].dt.quarter

    df["month_sin"] = np.sin(
        2 * np.pi * df["month_number"] / 12
    )

    df["month_cos"] = np.cos(
        2 * np.pi * df["month_number"] / 12
    )

    return df


# ============================================================
# 3. GENERATE LAG AND ROLLING FEATURES
# ============================================================

def add_demand_features(df, group_columns):
    df = df.copy()

    df = df.sort_values(
        group_columns + ["month"]
    ).reset_index(drop=True)

    grouped = df.groupby(group_columns)[TARGET]

    for lag in [1, 2, 3, 6, 12, 13]:
        df[f"lag_{lag}"] = grouped.shift(lag)

    for window in [3, 6, 12]:

        df[f"rolling_mean_{window}"] = (
            df.groupby(group_columns)[TARGET]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=window,
                    min_periods=window
                ).mean()
            )
        )

        df[f"rolling_std_{window}"] = (
            df.groupby(group_columns)[TARGET]
            .transform(
                lambda x: x.shift(1).rolling(
                    window=window,
                    min_periods=window
                ).std()
            )
        )

    return df


# Product features
product_panel = add_demand_features(
    product_panel,
    ["product_id"]
)

product_panel = add_calendar_features(product_panel)

product_panel = product_panel.merge(
    active_branch_table,
    on="month",
    how="left"
)

product_panel = product_panel.merge(
    products[
        [
            "product_id",
            "category_en",
            "audience",
            "listed_price_EGP"
        ]
    ].drop_duplicates("product_id"),
    on="product_id",
    how="left"
)


# SKU features
sku_panel = add_demand_features(
    sku_panel,
    ["product_id", "sku_id"]
)

sku_panel = add_calendar_features(sku_panel)

sku_panel = sku_panel.merge(
    active_branch_table,
    on="month",
    how="left"
)

sku_panel = sku_panel.merge(
    products[
        ["product_id", "category_en", "audience"]
    ].drop_duplicates("product_id"),
    on="product_id",
    how="left"
)


# ============================================================
# 4. EXTRACT TEST FEATURE MATRICES
# ============================================================

product_test_features = product_panel[
    product_panel["month"].isin(test_months)
].copy()

sku_test_features = sku_panel[
    sku_panel["month"].isin(test_months)
].copy()


# Validate required columns
missing_product_columns = [
    col for col in product_feature_columns
    if col not in product_test_features.columns
]

missing_sku_columns = [
    col for col in sku_feature_columns
    if col not in sku_test_features.columns
]

if missing_product_columns or missing_sku_columns:
    raise ValueError(
        f"Missing product columns: {missing_product_columns}\n"
        f"Missing SKU columns: {missing_sku_columns}"
    )


# Validate missing values
product_missing = (
    product_test_features[product_feature_columns]
    .isna().sum()
)

sku_missing = (
    sku_test_features[sku_feature_columns]
    .isna().sum()
)

if product_missing.sum() > 0 or sku_missing.sum() > 0:
    raise ValueError(
        "Missing values found in test features.\n"
        f"Product:\n{product_missing[product_missing > 0]}\n"
        f"SKU:\n{sku_missing[sku_missing > 0]}"
    )


# Match categorical data types
for col in product_categorical_features:
    product_test_features[col] = (
        product_test_features[col]
        .fillna("Unknown")
        .astype(str)
    )

for col in sku_categorical_features:
    sku_test_features[col] = (
        sku_test_features[col]
        .fillna("Unknown")
        .astype(str)
    )


X_product_test = product_test_features[
    product_feature_columns
].copy()

X_sku_test = sku_test_features[
    sku_feature_columns
].copy()


# ============================================================
# 5. GENERATE RAW MODEL PREDICTIONS
# ============================================================

product_predictions = product_test_features[
    ["product_id", "month"]
].copy()

product_predictions["product_forecast"] = (
    product_model.predict(X_product_test)
)

product_predictions["product_forecast"] = (
    product_predictions["product_forecast"].clip(lower=0)
)


sku_predictions = sku_test_features[
    ["product_id", "sku_id", "month"]
].copy()

sku_predictions["sku_forecast"] = (
    sku_model.predict(X_sku_test)
)

sku_predictions["sku_forecast"] = (
    sku_predictions["sku_forecast"].clip(lower=0)
)


# ============================================================
# 6. CALCULATE SKU RATIOS
# ============================================================

sku_predictions["total_sku_forecast"] = (
    sku_predictions
    .groupby(["product_id", "month"])["sku_forecast"]
    .transform("sum")
)

sku_predictions["sku_count"] = (
    sku_predictions
    .groupby(["product_id", "month"])["sku_id"]
    .transform("count")
)

sku_predictions["sku_ratio"] = np.where(
    sku_predictions["total_sku_forecast"] > 0,
    sku_predictions["sku_forecast"]
    / sku_predictions["total_sku_forecast"],
    1 / sku_predictions["sku_count"]
)


# ============================================================
# 7. ALLOCATE PRODUCT FORECAST TO SKUs
# ============================================================

allocated_sku_forecasts = sku_predictions.merge(
    product_predictions,
    on=["product_id", "month"],
    how="left"
)

allocated_sku_forecasts["allocated_sku_forecast"] = (
    allocated_sku_forecasts["product_forecast"]
    * allocated_sku_forecasts["sku_ratio"]
)


# ============================================================
# 8. PREPARE ACTUALS FOR EVALUATION
# ============================================================

product_evaluation = product_predictions.merge(
    product_test_actual[
        ["product_id", "month", TARGET]
    ],
    on=["product_id", "month"],
    how="left"
)

sku_evaluation = allocated_sku_forecasts.merge(
    sku_test_actual[
        ["product_id", "sku_id", "month", TARGET]
    ],
    on=["product_id", "sku_id", "month"],
    how="left"
)


# ============================================================
# 9. EVALUATION METRICS
# ============================================================

def calculate_metrics(actual, predicted):

    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    errors = predicted - actual

    mae = mean_absolute_error(actual, predicted)

    rmse = np.sqrt(
        mean_squared_error(actual, predicted)
    )

    wape = (
        np.sum(np.abs(errors)) / np.sum(actual) * 100
        if np.sum(actual) != 0
        else np.nan
    )

    bias = (
        np.sum(errors) / np.sum(actual) * 100
        if np.sum(actual) != 0
        else np.nan
    )

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
        "Bias (%)": bias
    }


# Product-level evaluation
product_metrics = calculate_metrics(
    product_evaluation[TARGET],
    product_evaluation["product_forecast"]
)

# Direct SKU-level evaluation
direct_sku_metrics = calculate_metrics(
    sku_evaluation[TARGET],
    sku_evaluation["sku_forecast"]
)

# Allocated SKU-level evaluation
allocated_sku_metrics = calculate_metrics(
    sku_evaluation[TARGET],
    sku_evaluation["allocated_sku_forecast"]
)


# ============================================================
# 10. FINAL COMPARISON
# ============================================================

evaluation_results = pd.DataFrame(
    [
        {
            "Model": "Product-Level",
            **product_metrics
        },
        {
            "Model": "Direct SKU-Level",
            **direct_sku_metrics
        },
        {
            "Model": "Allocated SKU-Level",
            **allocated_sku_metrics
        }
    ]
).set_index("Model")

print("FORECAST EVALUATION RESULTS")
print("=" * 70)

display(evaluation_results.round(3))


# ============================================================
# 11. VALIDATE ALLOCATION CONSERVATION
# ============================================================

allocation_check = (
    allocated_sku_forecasts
    .groupby(["product_id", "month"], as_index=False)
    .agg(
        allocated_total=("allocated_sku_forecast", "sum"),
        product_total=("product_forecast", "first")
    )
)

allocation_check["difference"] = (
    allocation_check["allocated_total"]
    - allocation_check["product_total"]
)

print("\nALLOCATION CONSERVATION CHECK")
print("=" * 70)

print(
    "Maximum absolute allocation difference:",
    allocation_check["difference"].abs().max()
)

print(
    "All allocations conserved:",
    np.allclose(
        allocation_check["allocated_total"],
        allocation_check["product_total"],
        atol=1e-8
    )
)

print("\nFinal datasets:")
print("Product predictions:", product_predictions.shape)
print("Direct SKU predictions:", sku_predictions.shape)
print("Allocated SKU forecasts:", allocated_sku_forecasts.shape)


FORECAST EVALUATION RESULTS


,MAE,RMSE,WAPE (%),Bias (%)
Model,,,,
Product-Level,84.38,113.63,11.83,-7.77
Direct SKU-Level,30.88,42.10,21.64,-6.51
Allocated SKU-Level,31.23,42.69,21.89,-7.77



ALLOCATION CONSERVATION CHECK
Maximum absolute allocation difference: 1.1368683772161603e-13
All allocations conserved: True

Final datasets:
Product predictions: (180, 3)
Direct SKU predictions: (900, 7)
Allocated SKU forecasts: (900, 9)
